Weeky Assessment -Agentic Ai and RAG

🔷 Question 2: Design a Multi-Agent Workflow with LangGraph (25 Marks)
🧩 Scenario
You are building an AI-powered customer support system for a fintech company.

The system must handle:

Transaction queries
Fraud detection flags
Refund requests
General FAQs
The company wants:

High accuracy
Step-by-step reasoning
Ability to retry if answer is incorrect
Modular, scalable architecture
💻 Task
Design and implement a multi-agent workflow using LangGraph (or similar framework).

✅ 1. Agent Design
Define at least 3 agents, such as:

Retrieval Agent
Reasoning/Answer Agent
Validation Agent
Explain briefly (in comments or code):

Each agent’s role
Input/output
✅ 2. Graph Workflow Implementation
Write code or pseudo-code to:

Define state
Add nodes (agents)
Define edges
Implement conditional logic
👉 Must include:

Retry loop if validation fails
Clear start and end states
✅ 3. State Management
Show how state evolves across steps:

Query
Context
Intermediate reasoning
Final answer
Validation flag
✅ 4. Task Delegation & Communication
Demonstrate:

How agents pass information
How decisions are made between agents
🎯 Expected Outcome
A clear multi-step, graph-based agent system that:

Handles complex queries
Demonstrates reasoning + validation
Uses proper orchestration

In [9]:
import os
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('groq_api_key')
except:
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")




# INSTALL DEPENDENCIES
!pip install -qqq langchain_text_splitters langchain_community langchain_core langchain_groq faiss-cpu sentence-transformers


# LLM
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",  # Updated Groq model as per user request
    temperature=0,
    groq_api_key=GROQ_API_KEY # Pass the API key here
)


# STATE DEFINITION
class AgentState(TypedDict):
    query: str
    context: Optional[str]
    reasoning: Optional[str]
    answer: Optional[str]
    valid: Optional[bool]
    retries: int


# AGENT 1: RETRIEVAL
def retrieval_agent(state: AgentState):
    """
    Role:
    - Retrieve relevant info (simulate RAG or DB lookup)

    Input: query
    Output: context
    """
    query = state["query"]

    # Simulated knowledge base
    knowledge_base = {
        "refund": "Refunds are processed within 5-7 business days.",
        "fraud": "If fraud is detected, account is temporarily blocked.",
        "transaction": "Transactions can take up to 24 hours to reflect."
    }

    context = "General FAQ: Contact support."

    for key in knowledge_base:
        if key in query.lower():
            context = knowledge_base[key]

    return {**state, "context": context}


# AGENT 2: REASONING
def reasoning_agent(state: AgentState):
    """
    Role:
    - Generate answer using reasoning

    Input: query + context
    Output: reasoning + answer
    """
    prompt = f"""
    Answer the question using the context.

    Context: {state['context']}
    Question: {state['query']}

    Provide:
    1. Reasoning
    2. Final Answer
    """

    response = llm.invoke(prompt).content

    return {
        **state,
        "reasoning": response,
        "answer": response
    }


# AGENT 3: VALIDATION
def validation_agent(state: AgentState):
    """
    Role:
    - Validate answer correctness

    Input: query + answer
    Output: valid (True/False)
    """
    prompt = f"""
    Check if the answer correctly addresses the query.

    Query: {state['query']}
    Answer: {state['answer']}

    Reply ONLY with True or False.
    """

    result = llm.invoke(prompt).content.strip()

    is_valid = "true" in result.lower()

    return {
        **state,
        "valid": is_valid,
        "retries": state["retries"] + 1
    }


# CONDITIONAL LOGIC
def should_retry(state: AgentState):
    """
    Decide whether to retry or end
    """
    if not state["valid"] and state["retries"] < 2:
        return "retry"
    return "end"


# BUILD GRAPH
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("retrieval", retrieval_agent)
builder.add_node("reasoning", reasoning_agent)
builder.add_node("validation", validation_agent)

# Define flow
builder.set_entry_point("retrieval")

builder.add_edge("retrieval", "reasoning")
builder.add_edge("reasoning", "validation")

# Conditional edge (retry loop)
builder.add_conditional_edges(
    "validation",
    should_retry,
    {
        "retry": "reasoning",  # retry reasoning
        "end": END
    }
)

# Compile graph
graph = builder.compile()


# RUN WORKFLOW
def run_system(query):
    initial_state = {
        "query": query,
        "context": None,
        "reasoning": None,
        "answer": None,
        "valid": False,
        "retries": 0
    }

    result = graph.invoke(initial_state)
    return result


# TEST
if __name__ == "__main__":
    output = run_system("How long does refund take?")

    print("\nFinal Answer:", output["answer"])
    print("Valid:", output["valid"])
    print("Retries:", output["retries"])


Final Answer: 1. Reasoning: The question asks about the time it takes for a refund to be processed. The context provides a specific time frame for refunds, which is within 5-7 business days. This information directly answers the question about the duration of the refund process.

2. Final Answer: 5-7 business days.
Valid: True
Retries: 1


In [ ]:
import nbformat
import os

# Set the notebook file names
notebook_name = "Que2MultiAgentWorkflowWithLangGraph.ipynb"
fixed_notebook_name = "Que2MultiAgentWorkflowWithLangGraph_fixed.ipynb"

if not os.path.exists(notebook_name):
    print(f"Error: The notebook file '{notebook_name}' was not found.")
else:
    with open(notebook_name, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    # Remove widget metadata if present
    if "widgets" in nb.get("metadata", {}):
        del nb["metadata"]["widgets"]

    with open(fixed_notebook_name, "w", encoding="utf-8") as f:
        nbformat.write(nb, f)

    print(f"Fixed notebook saved to '{fixed_notebook_name}'!")